# E042 — Speed TTA + Tied Covariance Combo

**Motivation:** E037 (tied cov) achieved 0.69% EER without speed TTA. E025+speedTTA achieved 1.67%. Combining tied covariance with speed TTA could push audio even lower.

**Hypothesis:** Speed TTA (original + 0.9x + 1.1x) with tied covariance UBM will improve over tied cov alone (0.69%).

**Configs:**
- `single`: E037 baseline (tied cov, no TTA)
- `speedTTA`: 3 views (0.9x, 1.0x, 1.1x)

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

import numpy as np
import librosa
from scipy.special import logsumexp
from sklearn.mixture import GaussianMixture
import copy

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

SEED = 67
UBM_COMPONENTS = 32
MAP_R = 16.0

DATA = Path('../data').resolve()
manifest = load_manifest(DATA)
y_all = manifest['label'].to_numpy()
print(f'{len(manifest)} samples')

E037_REF = {'mean_eer': 0.69, 'std_eer': 0.98}

222 samples


In [2]:
def _find_wav(stem, data_dir):
    for sf in ("target_train", "target_dev", "non_target_train", "non_target_dev"):
        p = data_dir / sf / (stem + ".wav")
        if p.exists(): return p
    raise FileNotFoundError(stem)

def _extract_lpcc(y, sr, order=12, n_cep=13, hop_length=160, win_length=400):
    frames = librosa.util.frame(y, frame_length=win_length, hop_length=hop_length)
    cep_frames = []
    for frame in frames.T:
        frame = frame * np.hanning(len(frame))
        try:
            a = librosa.lpc(frame.astype(np.float64), order=order)
            A_freq = np.fft.rfft(a, n=512)
            log_H = -np.log(np.abs(A_freq) + 1e-10)
            cep = np.real(np.fft.irfft(log_H))[:n_cep]
        except Exception:
            cep = np.zeros(n_cep)
        cep_frames.append(cep)
    feat = np.array(cep_frames, dtype=np.float32)
    delta = librosa.feature.delta(feat.T).T
    delta2 = librosa.feature.delta(feat.T, order=2).T
    feat = np.hstack([feat, delta, delta2])
    feat -= feat.mean(axis=0)
    return feat

def _aug_pitch(y, sr, rng):
    n_steps = float(rng.choice([-2, -1, 1, 2]))
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)

def _train_ubm(X, cov_type='tied'):
    return GaussianMixture(n_components=UBM_COMPONENTS, covariance_type=cov_type,
                           max_iter=200, random_state=SEED).fit(X)

def _map_adapt(ubm, X_target):
    log_resp = ubm._estimate_log_prob(X_target) + np.log(ubm.weights_)
    log_resp -= logsumexp(log_resp, axis=1, keepdims=True)
    resp = np.exp(log_resp)
    n_k = resp.sum(axis=0)
    mu_hat = (resp.T @ X_target) / (n_k[:, None] + 1e-10)
    alpha = n_k / (n_k + MAP_R)
    adapted = copy.deepcopy(ubm)
    adapted.means_ = alpha[:, None] * mu_hat + (1 - alpha[:, None]) * ubm.means_
    return adapted

def train_tied_model(train_df, data_dir, seed):
    rng = np.random.default_rng(seed)
    X_target, X_nontarget = [], []
    for _, row in train_df.iterrows():
        y_wav, sr = librosa.load(_find_wav(row["stem"], data_dir), sr=None, mono=True)
        wavs = [y_wav]
        if row["label"] == 1:
            wavs.append(_aug_pitch(y_wav, sr, rng))
        for y_aug in wavs:
            feat = _extract_lpcc(y_aug, sr)
            if row["label"] == 1:
                X_target.append(feat)
            else:
                X_nontarget.append(feat)
    ubm = _train_ubm(np.vstack(X_nontarget), cov_type='tied')
    adapted = _map_adapt(ubm, np.vstack(X_target))
    return ubm, adapted

def score_single(wav_path, adapted, ubm):
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    feat = _extract_lpcc(y, sr)
    return adapted.score(feat) - ubm.score(feat)

def score_speed_tta(wav_path, adapted, ubm):
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    scores = []
    for rate in [0.9, 1.0, 1.1]:
        y_stretched = librosa.effects.time_stretch(y, rate=rate)
        feat = _extract_lpcc(y_stretched, sr)
        scores.append(adapted.score(feat) - ubm.score(feat))
    return np.mean(scores)

print('Functions defined')

Functions defined


## 2. Cross-validation

In [3]:
scoring_fns = {'single (E037)': score_single, 'speedTTA': score_speed_tta}
results = {}

for name, score_fn in scoring_fns.items():
    print(f"\n=== {name} ===")
    fold_eers = []
    for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
        seed_f = SEED + fold_id
        train_df = manifest.loc[train_idx]
        val_df = manifest.loc[val_idx]
        ubm, adapted = train_tied_model(train_df, DATA, seed_f)
        scores = [score_fn(_find_wav(row["stem"], DATA), adapted, ubm) for _, row in val_df.iterrows()]
        scores = np.array(scores)
        eer, _ = compute_eer(scores[y_all[val_idx] == 1], scores[y_all[val_idx] == 0])
        fold_eers.append(eer * 100)
        print(f"  Fold {fold_id}: EER={eer*100:.2f}%")
    results[name] = {'fold_eers': fold_eers, 'mean': np.mean(fold_eers), 'std': np.std(fold_eers)}
    print(f"  Mean +/- Std: {np.mean(fold_eers):.2f} +/- {np.std(fold_eers):.2f}%")

print("\n=== Summary ===")
for name, res in results.items():
    imp = results['single (E037)']['mean'] - res['mean']
    print(f"{name:15s}: {res['mean']:5.2f} +/- {res['std']:5.2f}%  (improvement: {imp:+.2f}pp)")


=== single (E037) ===


  Fold 0: EER=2.08%


  Fold 1: EER=0.00%


  Fold 2: EER=0.00%
  Mean +/- Std: 0.69 +/- 0.98%

=== speedTTA ===


  Fold 0: EER=1.39%


  Fold 1: EER=0.00%


  Fold 2: EER=0.00%
  Mean +/- Std: 0.46 +/- 0.65%

=== Summary ===
single (E037)  :  0.69 +/-  0.98%  (improvement: +0.00pp)
speedTTA       :  0.46 +/-  0.65%  (improvement: +0.23pp)


## 3. Conclusion

Speed TTA + tied cov: [TBD]
Decision: [Adopt/Reject]